In [ ]:
import json
import pandas as pd

import re
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

In [ ]:
with open("../data/raw_jobs.json", "r", encoding="utf-8") as f:
    raw_jobs = json.load(f)

df = pd.DataFrame(raw_jobs)

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

In [ ]:
df_clean = df[[
    "id",
    "headline",
    "occupation",
    "occupation_group",
    "occupation_field",
    "employer",
    "workplace_address",
    "description",
    "must_have",
    "nice_to_have",
    "publication_date",
    "experience_required"
]].copy()

print(f"Reduced from {df.shape[1]} to {df_clean.shape[1]} columns")
df_clean.head(3)

In [ ]:
df_clean.info()

In [ ]:
# Flatten nested columns to directly access the content
df_clean["occupation_label"] = df_clean["occupation"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
df_clean["occupation_group_label"] = df_clean["occupation_group"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
df_clean["occupation_field_label"] = df_clean["occupation_field"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
df_clean["employer_name"] = df_clean["employer"].apply(lambda x: x.get("name") if isinstance(x, dict) else None)
df_clean["municipality"] = df_clean["workplace_address"].apply(lambda x: x.get("municipality") if isinstance(x, dict) else None)
df_clean["region"] = df_clean["workplace_address"].apply(lambda x: x.get("region") if isinstance(x, dict) else None)
df_clean["description_text"] = df_clean["description"].apply(lambda x: x.get("text") if isinstance(x, dict) else None)

# Drop the original nested columns
df_clean = df_clean.drop(columns=["occupation", "occupation_group", "occupation_field", 
                                   "employer", "workplace_address", "description"])

print(df_clean.shape)
df_clean.head(3)

In [ ]:
# There are a must_have and a nice_to_have sections, I want to check how many jobs have those sections populated with structured skills
has_must_have = df_clean["must_have"].apply(
    lambda x: len(x.get("skills", [])) > 0 if isinstance(x, dict) else False
).sum()

has_nice_to_have = df_clean["nice_to_have"].apply(
    lambda x: len(x.get("skills", [])) > 0 if isinstance(x, dict) else False
).sum()

print(f"Jobs with must_have skills: {has_must_have} / {len(df_clean)}")
print(f"Jobs with nice_to_have skills: {has_nice_to_have} / {len(df_clean)}")